In [1]:
# Run this in your first Colab cell
from google.colab import files
uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
# Install libraries
!pip install sqlalchemy pandas matplotlib seaborn plotly

import pandas as pd
import sqlite3
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

✅ Libraries loaded


In [3]:
# Load dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Shape: {df.shape}")
print(df.head())
print(df.dtypes)

Shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies   

In [4]:
# Create SQLite database and load data into it
engine = create_engine('sqlite:///churn.db')

df.to_sql('customers', engine, if_exists='replace', index=False)
print("✅ Data loaded into churn.db — table: customers")

✅ Data loaded into churn.db — table: customers


**SQL Analytics** ***- We'll run all SQL queries directly inside Google Colab using SQLite + pandas. No separate SQL client needed.***


In [5]:
# Step 1: Setup — Load Data into SQLite
# Imports & DB connection
import pandas as pd
import sqlite3
from sqlalchemy import create_engine, text

# Load CSV
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Fix column names → lowercase + underscores (cleaner for SQL)
df.columns = df.columns.str.lower().str.replace(' ', '_', regex=False)

# TotalCharges is stored as string in the raw file — fix it
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')

# Create SQLite DB and load table
engine = create_engine('sqlite:///churn.db')
df.to_sql('customers', engine, if_exists='replace', index=False)

print(f"✅ Table 'customers' created with {len(df)} rows")
print(f"Columns: {list(df.columns)}")

✅ Table 'customers' created with 7043 rows
Columns: ['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']


In [6]:
# Step 2: Helper Function
# Reusable query runner
def run_query(sql):
    """Run a SQL query and return a styled pandas DataFrame."""
    with engine.connect() as conn:
        result = pd.read_sql(text(sql), conn)
    return result

print("✅ run_query() ready")

✅ run_query() ready


In [7]:
# Step 3: Schema Inspection

# Inspect the table
run_query("PRAGMA table_info(customers)")

# Row count
run_query("SELECT COUNT(*) AS total_customers FROM customers")

,total_customers
0,7043


In [8]:
# Step 4: Core Business Queries

# Query 1 — Overall Churn Rate
churn_rate = run_query("""
    SELECT
        COUNT(*)                                         AS total_customers,
        SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)  AS churned_customers,
        ROUND(
            SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        )                                                AS churn_rate_pct
    FROM customers
""")
print(churn_rate)

   total_customers  churned_customers  churn_rate_pct
0             7043               1869           26.54


In [9]:
# Query 2 — Revenue Loss from Churned Customers
revenue_loss = run_query("""
    SELECT
        ROUND(SUM(monthlycharges), 2)   AS total_monthly_revenue,
        ROUND(SUM(CASE WHEN churn = 'Yes' THEN monthlycharges ELSE 0 END), 2)
                                         AS monthly_revenue_lost,
        ROUND(SUM(CASE WHEN churn = 'Yes' THEN totalcharges  ELSE 0 END), 2)
                                         AS total_revenue_lost
    FROM customers
""")
print(revenue_loss)

   total_monthly_revenue  monthly_revenue_lost  total_revenue_lost
0               456116.6             139130.85           2862926.9


In [18]:
# Query 3 — Churn by Contract Type
churn_by_contract = run_query("""
    SELECT
        contract,
        COUNT(*)  AS total,
        SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)     AS churned,
        ROUND(
            SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        )                                                     AS churn_rate_pct
    FROM customers
    GROUP BY contract
    ORDER BY churn_rate_pct DESC
""")
print(churn_by_contract)

         contract  total  churned  churn_rate_pct
0  Month-to-month   3875     1655           42.71
1        One year   1473      166           11.27
2        Two year   1695       48            2.83


In [17]:
# Query 4 — Churn by Internet Service
churn_by_internet = run_query("""
    SELECT
        internetservice,
        COUNT(*) AS total,
        SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)  AS churned,
        ROUND(
            SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        )                                                     AS churn_rate_pct
    FROM customers
    GROUP BY internetservice
    ORDER BY churn_rate_pct DESC
""")
print(churn_by_internet)

  internetservice  total  churned  churn_rate_pct
0     Fiber optic   3096     1297           41.89
1             DSL   2421      459           18.96
2              No   1526      113            7.40


In [16]:
# Query 5 — Churn by Payment Method
churn_by_payment = run_query("""
    SELECT
        paymentmethod,
        COUNT(*)                                         AS total,
        SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END)   AS churned,
        ROUND(
            SUM(CASE WHEN churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
        )                         AS churn_rate_pct
    FROM customers
    GROUP BY paymentmethod
    ORDER BY churn_rate_pct DESC
""")
print(churn_by_payment)

               paymentmethod  total  churned  churn_rate_pct
0           Electronic check   2365     1071           45.29
1               Mailed check   1612      308           19.11
2  Bank transfer (automatic)   1544      258           16.71
3    Credit card (automatic)   1522      232           15.24


In [13]:
# Query 6 — Average Monthly Charges (Churned vs Retained)
avg_charges = run_query("""
    SELECT
        churn,
        ROUND(AVG(monthlycharges), 2)  AS avg_monthly_charges,
        ROUND(AVG(totalcharges),   2)  AS avg_total_charges,
        ROUND(AVG(tenure),         2)  AS avg_tenure_months
    FROM customers
    GROUP BY churn
""")
print(avg_charges)

  churn  avg_monthly_charges  avg_total_charges  avg_tenure_months
0    No                61.27            2555.34              37.57
1   Yes                74.44            1531.80              17.98


In [14]:
# Query 7 — High-Risk Customer Segment
high_risk = run_query("""
    SELECT
        customerid,
        contract,
        tenure,
        monthlycharges,
        internetservice,
        paymentmethod
    FROM customers
    WHERE
        churn        = 'No'
        AND contract = 'Month-to-month'
        AND tenure   < 12
        AND monthlycharges > 65
    ORDER BY monthlycharges DESC
    LIMIT 20
""")
print(f"High-risk customers (likely to churn next): {len(high_risk)}")
print(high_risk)

High-risk customers (likely to churn next): 20
    customerid        contract  tenure  monthlycharges internetservice  \
0   3292-PBZEJ  Month-to-month      11          111.40     Fiber optic   
1   5760-IFJOZ  Month-to-month       3          107.95     Fiber optic   
2   2081-VEYEH  Month-to-month       3          107.95     Fiber optic   
3   6734-GMPVK  Month-to-month       5          105.30     Fiber optic   
4   2018-PZKMU  Month-to-month       9          103.10     Fiber optic   
5   9547-ITEFG  Month-to-month       9          102.60     Fiber optic   
6   8118-TJAFG  Month-to-month       9          101.50     Fiber optic   
7   4132-KALRO  Month-to-month       4          100.85     Fiber optic   
8   0722-SVSFK  Month-to-month       7          100.40     Fiber optic   
9   7379-FNIUJ  Month-to-month       2          100.20     Fiber optic   
10  2754-SDJRD  Month-to-month       8          100.15     Fiber optic   
11  4445-ZJNMU  Month-to-month       9           99.30     Fiber 